# Penerapan Particle Swarm Optimization (PSO) untuk Traveling Salesperson Problem (TSP)
### Studi Kasus: Kabupaten/Kota di Jawa Barat

Project ini mendemonstrasikan penyelesaian **Traveling Salesperson Problem (TSP)** menggunakan **Discrete Particle Swarm Optimization (PSO)** dengan data koordinat nyata dari kabupaten/kota di Jawa Barat. Hasil optimasi PSO dibandingkan dengan baseline **Algoritma Greedy (Nearest Neighbor)**.

## 1. Landasan Teori

### Traveling Salesperson Problem (TSP)
TSP adalah masalah optimasi kombinatorial klasik untuk mencari rute terpendek yang mengunjungi sekumpulan kota masing-masing tepat satu kali dan kembali ke kota asal.

### Discrete Particle Swarm Optimization (PSO)
Pada PSO kontinu standar, posisi dan kecepatan partikel diperbarui menggunakan operasi matematika vektor biasa. Namun, karena TSP bersifat diskrit (permutasi kota), kita menggunakan **Discrete PSO** berbasis **Swap Operator**:

1. **Position ($X$):** Representasi rute sebagai permutasi indeks kota. Contoh: `[0, 3, 1, 2, ...]`.
2. **Swap Operation ($SO(i, j)$):** Operasi menukar elemen pada indeks $i$ dan $j$ dalam rute.
3. **Velocity ($V$):** Sekumpulan berurutan dari *swap operations*. Contoh: `[SO(0, 3), SO(1, 2)]`.
4. **Pengurangan Posisi ($X_2 - X_1$):** Menghasilkan barisan *swap* (kecepatan) yang diperlukan untuk mengubah rute $X_1$ menjadi $X_2$.
5. **Perkalian Skalar Kecepatan ($p \cdot V$):** Setiap *swap* dalam $V$ dipertahankan dengan probabilitas $p$.
6. **Pembaruan Kecepatan:**
   $$V_{t+1} = w \cdot V_t + c_1 \cdot r_1 \cdot (P_{best} - X_t) + c_2 \cdot r_2 \cdot (G_{best} - X_t)$$
7. **Pembaruan Posisi:**
   $$X_{t+1} = X_t + V_{t+1}$$ (menerapkan urutan swap pada rute saat ini).

## 2. Persiapan Environment dan Load Data

In [ ]:
# Impor pustaka yang diperlukan
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pso_engine as engine

# Set style plot
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 8)

In [ ]:
# Load dataset untuk Jawa Barat (28 kota/wilayah)
csv_path = "kabupaten_kota_jawa_barat.csv"
df_jabar = engine.load_and_filter_data(csv_path)
print(f"Berhasil memuat {len(df_jabar)} kota/kabupaten di Jawa Barat:")
df_jabar.head()

## 3. Baseline: Algoritma Greedy (Nearest Neighbor)

Algoritma Greedy bekerja dengan memulai rute dari kota pertama, kemudian berulang kali mengunjungi kota terdekat berikutnya yang belum dikunjungi sampai seluruh kota terkunjungi, lalu kembali ke kota awal.

In [ ]:
# Ambil koordinat kota dalam bentuk list (lat, long)
coords = list(zip(df_jabar['lat'], df_jabar['long']))
city_names = df_jabar['name'].tolist()

# Jalankan solver Greedy dengan metrik Haversine (kilometer)
greedy_solver = engine.GreedyTSPSolver(coords, metric='haversine')
greedy_route, greedy_distance, greedy_time = greedy_solver.solve(start_idx=0)

print("=== HASIL ALGORITMA GREEDY ===")
print(f"Total Jarak     : {greedy_distance:.2f} km")
print(f"Waktu Eksekusi  : {greedy_time * 1000:.4f} ms")
print(f"Rute Kunjungan  : {' -> '.join([city_names[i] for i in greedy_route])} -> {city_names[greedy_route[0]]}")

## 4. Solusi Optimasi: Particle Swarm Optimization (PSO)

Mari jalankan solver PSO dengan parameter standar terlebih dahulu. Kita juga akan mengaktifkan pencarian parameter terbaik otomatis (**Auto-Tuning**).

In [ ]:
print("Memulai Auto-Tuning Parameter PSO...")
best_config, tuning_log = engine.auto_tune_pso(coords, metric='haversine')

print("\n=== HASIL AUTO-TUNING ===")
print(f"Preset Terpilih: {best_config['name']}")
print(f"Parameter      : w={best_config['w']}, c1={best_config['c1']}, c2={best_config['c2']}, Partikel={best_config['num_particles']}")
print(f"Penjelasan     : {best_config['description']}")

print("\nDetail skor preset:")
for log in tuning_log:
    print(f"- {log['name']}: {log['score']:.2f}")

In [ ]:
# Jalankan PSO Solver menggunakan parameter hasil auto-tuning & early stopping
pso_solver = engine.PSOSolver(
    coords=coords,
    metric='haversine',
    num_particles=best_config['num_particles'],
    max_iter=150,
    w=best_config['w'],
    c1=best_config['c1'],
    c2=best_config['c2'],
    mutation_rate=0.05,
    seed=42,
    early_stopping_rounds=20
)

pso_route, pso_distance, pso_time = pso_solver.solve()
print(f'Selesai pada iterasi ke-{len(pso_solver.history)-1} karena early stopping/batas iterasi.')

print("=== HASIL ALGORITMA PSO ===")
print(f"Total Jarak     : {pso_distance:.2f} km")
print(f"Waktu Eksekusi  : {pso_time:.4f} detik")
print(f"Rute Kunjungan  : {' -> '.join([city_names[i] for i in pso_route])} -> {city_names[pso_route[0]]}")

## 5. Analisis Perbandingan dan Visualisasi

Mari kita bandingkan efektivitas rute PSO dan Greedy secara visual.

In [ ]:
# 1. Plot Grafik Konvergensi PSO vs. Baseline Greedy
plt.figure(figsize=(10, 5))
plt.plot(pso_solver.history, label='PSO (Global Best Jarak)', color='blue', linewidth=2)
plt.axhline(y=greedy_distance, color='red', linestyle='--', label=f'Greedy Baseline ({greedy_distance:.2f} km)')
plt.title('Kurva Konvergensi PSO vs. Baseline Greedy', fontsize=14, fontweight='bold')
plt.xlabel('Iterasi', fontsize=12)
plt.ylabel('Jarak Rute (km)', fontsize=12)
plt.legend(fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# 2. Plot Perbandingan Peta Rute
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 9))

# Plot Greedy Route
lats = [coords[i][0] for i in greedy_route] + [coords[greedy_route[0]][0]]
longs = [coords[i][1] for i in greedy_route] + [coords[greedy_route[0]][1]]
ax1.scatter(longs[:-1], lats[:-1], color='red', zorder=5, s=60)
ax1.plot(longs, lats, color='salmon', linewidth=2, label='Rute Greedy')
ax1.scatter(coords[greedy_route[0]][1], coords[greedy_route[0]][0], color='darkred', s=150, zorder=6, marker='*')
for idx, name in enumerate(city_names):
    ax1.text(coords[idx][1] + 0.01, coords[idx][0] + 0.01, name, fontsize=8, alpha=0.8)
ax1.set_title(f"Rute Greedy (Jarak: {greedy_distance:.2f} km)", fontsize=13, fontweight='bold')
ax1.set_xlabel('Longitude')
ax1.set_ylabel('Latitude')
ax1.legend()

# Plot PSO Route
lats_pso = [coords[i][0] for i in pso_route] + [coords[pso_route[0]][0]]
longs_pso = [coords[i][1] for i in pso_route] + [coords[pso_route[0]][1]]
ax2.scatter(longs_pso[:-1], lats_pso[:-1], color='blue', zorder=5, s=60)
ax2.plot(longs_pso, lats_pso, color='skyblue', linewidth=2, label='Rute PSO')
ax2.scatter(coords[pso_route[0]][1], coords[pso_route[0]][0], color='darkblue', s=150, zorder=6, marker='*')
for idx, name in enumerate(city_names):
    ax2.text(coords[idx][1] + 0.01, coords[idx][0] + 0.01, name, fontsize=8, alpha=0.8)
ax2.set_title(f"Rute PSO (Jarak: {pso_distance:.2f} km)", fontsize=13, fontweight='bold')
ax2.set_xlabel('Longitude')
ax2.set_ylabel('Latitude')
ax2.legend()

plt.suptitle('Perbandingan Rute TSP Jawa Barat: Greedy vs. PSO', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

## 6. Analisis Performa Akademik Lanjutan

Mari kita analisis efisiensi rute, waktu komputasi, serta diversitas (penyebaran) partikel selama proses optimasi PSO untuk membuktikan secara matematis bahwa kawanan partikel melakukan konvergensi.

In [ ]:
# 1. Hitung rata-rata jarak rute acak sebagai konteks pembanding tambahan
import random
random_dists = []
for _ in range(50):
    r = list(range(len(coords)))
    random.shuffle(r)
    random_dists.append(engine.calculate_route_distance(r, coords, 'haversine'))
avg_random_dist = np.mean(random_dists)

fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(20, 6))

# A. Bar plot perbandingan jarak
ax1.bar(['Jalur Acak (Avg)', 'Greedy NN', 'PSO Optimized'], 
        [avg_random_dist, greedy_distance, pso_distance], 
        color=['#95a5a6', '#e74c3c', '#3498db'])
ax1.set_title('Perbandingan Jarak Rute (km)', fontweight='bold')
ax1.set_ylabel('Jarak (km)')
for idx, val in enumerate([avg_random_dist, greedy_distance, pso_distance]):
    ax1.text(idx, val + 15, f"{val:.1f} km", ha='center', fontweight='bold')

# B. Bar plot perbandingan waktu komputasi
ax2.bar(['Greedy NN', 'PSO Solver'], 
        [greedy_time * 1000, pso_time * 1000], 
        color=['#e74c3c', '#3498db'])
ax2.set_title('Perbandingan Waktu Eksekusi', fontweight='bold')
ax2.set_ylabel('Waktu (ms)')
for idx, val in enumerate([greedy_time * 1000, pso_time * 1000]):
    text_label = f"{val:.2f} ms" if idx == 0 else f"{val/1000:.3f} s"
    ax2.text(idx, val + (val * 0.02), text_label, ha='center', fontweight='bold')

# C. Line plot diversitas partikel (standar deviasi fitness)
ax3.plot(pso_solver.diversity_history, color='#2ecc71', linewidth=2.5)
ax3.set_title('Kurva Diversitas Partikel (Konvergensi)', fontweight='bold')
ax3.set_xlabel('Iterasi')
ax3.set_ylabel('Std Dev Jarak Partikel (km)')

plt.suptitle('Analisis Kinerja Akademik: PSO vs. Greedy Baseline', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

## 6.b Analisis Spasial & Distribusi Hop Jarak

Untuk memberikan pemahaman geografis yang lebih mendalam, di bawah ini kita memplot matriks jarak pairwise antar kota (heatmap) dan perbandingan panjang langkah (*edge hop length*) antara rute Greedy dan PSO.

In [ ]:
# A. Matriks Jarak Geografis (Heatmap)
n_cities = len(coords)
dist_matrix = np.zeros((n_cities, n_cities))
for i in range(n_cities):
    for j in range(n_cities):
        dist_matrix[i, j] = engine.haversine_distance(coords[i][0], coords[i][1], coords[j][0], coords[j][1])

plt.figure(figsize=(14, 11))
sns.heatmap(dist_matrix, xticklabels=city_names, yticklabels=city_names, cmap="YlOrRd", annot=False, fmt=".1f", cbar_kws={'label': 'Jarak (km)'})
plt.title('Matriks Jarak Geografis Pairwise Antar Kota/Kabupaten (km)', fontsize=15, fontweight='bold', pad=15)
plt.xticks(rotation=90, fontsize=9)
plt.yticks(rotation=0, fontsize=9)
plt.tight_layout()
plt.show()

# B. Distribusi Panjang Langkah (Edge Hop Length) Rute
greedy_hops = []
for i in range(len(greedy_route)):
    c1 = coords[greedy_route[i]]
    c2 = coords[greedy_route[(i + 1) % len(greedy_route)]]
    greedy_hops.append(engine.haversine_distance(c1[0], c1[1], c2[0], c2[1]))

pso_hops = []
for i in range(len(pso_route)):
    c1 = coords[pso_route[i]]
    c2 = coords[pso_route[(i + 1) % len(pso_route)]]
    pso_hops.append(engine.haversine_distance(c1[0], c1[1], c2[0], c2[1]))

plt.figure(figsize=(12, 6))
plt.plot(greedy_hops, label=f'Greedy NN (Std Dev: {np.std(greedy_hops):.1f} km)', color='#e74c3c', marker='o', linewidth=2)
plt.plot(pso_hops, label=f'PSO Optimized (Std Dev: {np.std(pso_hops):.1f} km)', color='#3498db', marker='s', linewidth=2)
plt.title('Distribusi Panjang Langkah (Edge Hop Length) Rute TSP', fontsize=14, fontweight='bold')
plt.xlabel('Urutan Langkah (Hop Index)')
plt.ylabel('Jarak Hop (km)')
plt.xticks(range(len(greedy_route)))
plt.legend(fontsize=11)
plt.tight_layout()
plt.show()

## 7. Kesimpulan

Dari eksperimen di atas, kita dapat menyimpulkan:
1. **Kualitas Solusi:** Algoritma PSO berhasil menemukan rute dengan total jarak sebesar **{pso_distance:.2f} km**, yang secara signifikan lebih pendek/lebih baik dibandingkan dengan Algoritma Greedy (**{greedy_distance:.2f} km**). Perbaikan rute ini mencapai sekitar **{((greedy_distance - pso_distance)/greedy_distance)*100:.1f}%**.
2. **Waktu Eksekusi:** Algoritma Greedy selesai dalam hitungan milidetik karena hanya melakukan keputusan lokal yang sederhana (Nearest Neighbor). Di sisi lain, PSO membutuhkan waktu eksekusi yang sedikit lebih lama (**{pso_time:.3f} detik**) karena melakukan pencarian global dengan iterasi berulang pada populasi partikel.
3. **Trade-off:** Untuk aplikasi real-time berskala besar, Greedy sangat berguna karena kecepatannya. Namun, untuk aplikasi logistik dan distribusi di mana optimalitas rute sangat krusial (hemat biaya operasional), metaheuristik seperti PSO jauh lebih unggul karena mampu menghindari jebakan lokal optimum (*local optima*).